In [8]:
import os 
from dotenv import load_dotenv

load_dotenv()

# OpenAI client reads OPENAI_API_KEY (not OPEN_API_KEY)
_openai_key = os.getenv("OPENAI_API_KEY") or os.getenv("OPEN_API_KEY")
if _openai_key:
    os.environ["OPENAI_API_KEY"] = _openai_key

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"



In [ ]:
## create a datapoints 
## that;s how i instertthe data in langsmith


from langsmith import Client
from langsmith.utils import LangSmithConflictError

client = Client()

## Define the dataset - these are my test data
dataset_name = "MychatBot"

try:
    dataset = client.create_dataset(dataset_name)
except LangSmithConflictError:
    dataset = client.read_dataset(dataset_name=dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples=[

    {"inputs": {"question": "What is this?"}, "outputs": {"answer": "This is a chatbot designed to assist you with your queries."}},
    {"inputs": {"question": "Who created you?"}, "outputs": {"answer": "I was created by a team of developers to help answer questions."}},
    {"inputs": {"question": "How can you help me?"}, "outputs": {"answer": "I can provide information, answer questions, and help you with various tasks."}},
    {"inputs": {"question": "What technologies do you use?"}, "outputs": {"answer": "I use natural language processing and machine learning to understand and respond to your questions."}},
    {"inputs": {"question": "Are you always available?"}, "outputs": {"answer": "Yes, I am available 24/7 to assist you."}},

    ]
)



{'example_ids': ['0dedbade-b0c7-4200-962d-2db37e671d25',
  '8fe5da82-636a-432c-8653-fd199735e5d1',
  '178ca48c-5b7e-422a-a618-900f05d5fce6',
  '762f8c38-6fd1-4a67-ab12-290a3d9408b1',
  '7b1fc147-d9d0-47f1-a9c5-162f772cc7e5'],
 'count': 5,
 'as_of': '2026-03-28T20:16:04.945657179Z'}

In [21]:
### Define metrics (LLm as a judge)

import openai

from langsmith import wrappers

openai_client = wrappers.wrap_openai(openai.OpenAI())

instruction = (
    "You grade chatbot answers against a reference. "
    "Answer YES if the chatbot response is correct and helpful for the question, "
    "including when it is semantically equivalent to the reference or adds reasonable detail. "
    "Answer NO only if it is wrong, misleading, refuses unreasonably, or misses the intent."
)

def _pred_answer(outputs: dict) -> str:
    return str(outputs.get("answer") or outputs.get("response") or "")


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = (
        f"Input Question: {inputs['question']}\n"
        f"Chatbot Answer: {_pred_answer(outputs)}\n"
        f"Reference Answer: {reference_outputs['answer']}\n\n"
        "Reply with only YES or NO."
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": instruction},
            {"role": "user", "content": user_content},
        ],
    ).choices[0].message.content

    text = (response or "").strip().upper()
    return text.startswith("Y")



In [22]:
# Concision: 0–1 score — 1.0 if as short or shorter than reference, else decays as the answer gets wordier
# (A strict bool `len(pred) <= len(ref)` is almost always False for real models vs short gold answers.)


def concision(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    pred = str(outputs.get("answer") or outputs.get("response") or "")
    ref = str(reference_outputs.get("answer", ""))
    pred_n = max(len(pred.split()), 1)
    ref_n = max(len(ref.split()), 1)
    return float(min(1.0, ref_n / pred_n))

In [23]:
### how to run the evaluation 

default_instruction = "Please evaluate the answers generated by the chatbot for correctness and helpfulness."

def my_app(question:str, model:str = "gpt-4o-mini", instruction:str = default_instruction)-> str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instruction},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content



In [24]:
### Call my app for every datapoints 

def ls_target(inputs: dict) -> dict:
    # Use "answer" so it matches dataset reference_outputs and evaluators
    return {"answer": my_app(inputs["question"])}

In [25]:
experiment_results = client.evaluate(
    ls_target,
    data= dataset_name,
    evaluators = [correctness, concision],
    experiment_prefix = 'openai-4o-mini-chatbot'
)

View the evaluation results for experiment: 'openai-4o-mini-chatbot-bdeb98a4' at:
https://smith.langchain.com/o/625709e1-aba8-489e-9216-414289067b1e/datasets/b8f87aeb-de1e-4e53-8ff9-b443192f2569/compare?selectedSessions=3ab2365a-719e-43d2-9e8d-ef4354940b57




5it [00:39,  7.93s/it]


In [32]:
## EVlaution for RAG

import os

# WebBaseLoader uses urllib; set a User-Agent to avoid warnings and some blocked requests
os.environ.setdefault(
    "USER_AGENT",
    "Mozilla/5.0 (compatible; RagEvaluation/1.0; +https://github.com/)",
)

from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


urls= [
"https://cloud.google.com/discover/what-are-ai-agents",
"https://docs.agent.ai/welcome",
"https://www.ibm.com/think/topics/ai-agents",
]

## Load documnets from tehe URL 

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]


## initialize a text sploitter with specific chunk size and overlap 

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

doc_splits = text_splitter.split_documents(docs_list)

vectorstore = InMemoryVectorStore.from_documents(doc_splits, OpenAIEmbeddings())

retriever = vectorstore.as_retriever(k=6)



In [33]:
retriever.invoke("what is Ai agent")

[Document(id='a82b02e9-1afc-47d0-80be-fbff2fad8347', metadata={'source': 'https://docs.agent.ai/welcome', 'title': 'Welcome - Agent.ai Documentation', 'language': 'en'}, page_content='Agent.ai is the #1 Professional Marketplace For AI Agents (also, the only professional marketplace for AI agents). It is a marketplace and professional network for AI agents and the people who love them. Here, you can discover, connect with and hire AI agents to do useful things.\nFor UsersDiscover, connect with and hire AI agents to do useful thingsFor BuildersBuild advanced AI agents using an easy, extensible, no-code platform with data tools and access to frontier LLMS.\n\u200bDo I have to be a developer to build AI agents?\nNot at all! Our platform is a no-code platform, where you can drag and drop various components together to build AI agents.\nOur builder has dozens of actions that can grab data from various data sources (i.e. X. Bluesky, LinkedIn, Google) and use any frontier LLM (i.e. OpenAI’s 4o

In [37]:
from langchain.chat_models import init_chat_model
llm = init_chat_model("openai:gpt-4o-mini")
llm

ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x13aeb6350>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x13aeb6210>, root_client=<openai.OpenAI object at 0x13aeb5e50>, root_async_client=<openai.AsyncOpenAI object at 0x139811e00>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [38]:
from langsmith import traceable 

# Add decorator 

@traceable
def rag_bot(questions: str)-> dict:
    docs = retriever.invoke(questions)
    docs_strings = " ".join(doc.page_content for doc in docs)
    # Construct an instruction for the LLM to analyze information like a helpful assistant.
    prompt = (
        "You are a helpful assistant who is very good at analyzing information. "
        "Given the following context, answer the user's question thoroughly and clearly:\n\n"
        f"Context:\n{docs_strings}\n\n"
        f"Question:\n{questions}\n\n"
        "Provide your answer with helpful reasoning and, if appropriate, cite relevant parts of the context."
    )

    # llm invoke 

    ai_msg = llm.invoke([
        {"role": "system", "content": prompt},
        {"role": "user", "content": questions},
    ])

    return {"answer": ai_msg.content, "documents": docs}







In [39]:
rag_bot("what is agent??")

{'answer': 'An "agent" refers to an entity that performs tasks or functions autonomously or semi-autonomously, often utilizing artificial intelligence (AI) to execute specific roles within various environments. Agents can be categorized based on their interactions, functionalities, and the number of agents working together.\n\n1. **Types of Agents Based on Interaction:**\n   - **Interactive Partners (Surface Agents):** These engage directly with users by assisting with tasks such as customer service, providing personalized support, handling inquiries, and facilitating transactions. They are user-triggered and often include conversational agents that engage in Q&A and interactions with humans.\n   - **Autonomous Background Processes (Background Agents):** These operate in the background, automating routine tasks, analyzing data, optimizing processes, and identifying issues without direct human interaction. They perform their functions driven by events or fulfill queued tasks.\n\n2. **Fu

In [44]:
### Dataset (RAG eval examples)

from langsmith import Client
from langsmith.utils import LangSmithConflictError

client = Client()

rag_dataset_name = "Sahil Dataset name"

examples = [
    {
        "inputs": {"question": "What are AI agents and how do they differ from AI assistants?"},
        "outputs": {"answer": "AI agents are autonomous or semi-autonomous entities that perform tasks, often using artificial intelligence. They can interact directly with users, perform analytical tasks, and collaborate with other agents. AI assistants are a type of AI agent focused on user interaction and task support. Agents may also operate independently in background roles, whereas assistants typically interact more visibly with users."},
    },
    {
        "inputs": {"question": "What is the difference between single agent and multi-agent systems?"},
        "outputs": {"answer": "A single agent system features one agent operating independently to achieve defined goals, often handling a specific task. Multi-agent systems involve multiple agents collaborating or competing to solve complex problems. These systems leverage the diverse abilities of different agents and can simulate communication and teamwork similar to human scenarios."},
    },
    {
        "inputs": {"question": "What are some types of agents based on their interaction?"},
        "outputs": {"answer": "Types of agents based on interaction include interactive partners (surface agents) that work directly with users, and analytical agents that process information and optimize processes, often without direct user communication."},
    },
    {
        "inputs": {"question": "Why are multi-agent systems important in AI?"},
        "outputs": {"answer": "Multi-agent systems are important because they allow for collaboration and division of complex tasks among multiple agents, increasing efficiency and enabling solutions to problems that are too complex for a single agent to handle alone."},
    },
]

try:
    dataset = client.create_dataset(rag_dataset_name)
except LangSmithConflictError:
    dataset = client.read_dataset(dataset_name=rag_dataset_name)

# Re-running this cell appends duplicate examples each time; skip if you only need the dataset to exist
client.create_examples(dataset_id=dataset.id, examples=examples)



{'example_ids': ['26fc7ee5-2991-4533-943b-8fc86d5ffd00',
  'f90bb2f8-35c3-4636-89f0-783ad5b281d5',
  'ac96e0d6-cf4e-422e-917d-0bc72c15cec9',
  '72f11b5f-e817-438e-8fea-509250c9147d'],
 'count': 4,
 'as_of': '2026-03-28T21:30:40.33118479Z'}

### EVALUATORS 

In [53]:
###Evaluators
#
# 4 ways to do it

# 1- coorectness response vs reference answers

from typing_extensions import Annotated, TypedDict

class CoorectnessGrade(TypedDict):
    explanation:Annotated[str, "Explain you reasoning for the score"]
    correct:Annotated[bool, "true if the answer is correct, False otherwise"]

    # correctness prompt 
correctness_prompt = """
You are a teacher grading a student's quiz answer. You will be given:
- The quiz question.
- The student's answer.
- The reference (ideal) answer.

Compare the student's answer to the reference answer. Decide if the student's answer is correct. Mark it as correct only if it contains all the essential facts and logic presented in the reference answer, and does not introduce critical errors or misconceptions. Be fair but rigorous, as in a classroom quiz.

Return your evaluation in the following JSON format:
{
  "correct": true or false,
  "explanation": "Briefly explain why the answer is correct or not, referring to the key points from the reference answer."
}
"""

grader_llm = init_chat_model(model="gpt-4o-mini", temperature =0).with_structured_output(CoorectnessGrade, method='json_schema', strict= True)



def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for Rag Answer Accuracy"""
    answers = f"""\
        QUESTION : {inputs['question']}
        GROUND TRUTH ANSWER: {reference_outputs['answer']}
        STUDENT ANSWER: {outputs['answer']}
        """
    grader = grader_llm.invoke([
        {"role": "system", "content": correctness_prompt},
        {"role": "user", "content": answers},
    ])
    return grader['correct']






In [54]:
#relevance : response vs Input 

class RelevanceGrade(TypedDict):
    explanation: Annotated[str, "Explain your reasoning for the score"]
    relevant: Annotated[bool, "true if the answer is relevant to the question, False otherwise"]

relevance_prompt = """
You are a teacher grading the relevance of a chatbot's answer to a question.
You will be given:
- The user's question.
- The chatbot's answer.

Determine if the chatbot's answer directly responds to the question and stays on topic.
Mark it as relevant only if the answer meaningfully addresses the question asked, even if it is not fully correct. Irrelevant answers are those that ignore the question, go off-topic, or offer generic unrelated content.

Return your evaluation in the following JSON format:
{
  "relevant": true or false,
  "explanation": "Briefly explain why the answer is relevant or not, referring to specific points in the answer and question."
}
"""

relevance_llm = init_chat_model(model="gpt-4o-mini", temperature=0).with_structured_output(RelevanceGrade, method='json_schema', strict=True)

def relevance_evaluator(inputs: dict, outputs: dict, reference_outputs: dict = None) -> bool:
    """An evaluator for answer relevance to the question"""
    data = f"""\
        QUESTION: {inputs['question']}
        CHATBOT ANSWER: {outputs['answer']}
    """
    grader = relevance_llm.invoke([
        {"role": "system", "content": relevance_prompt},
        {"role": "user", "content": data},
    ])
    return grader['relevant']







In [55]:
### gorundness : Response vs retrieved docs 

from typing import TypedDict, Annotated

class GroundednessGrade(TypedDict):
    grounded: Annotated[bool, "true if the answer is supported by the retrieved documents, False otherwise"]
    explanation: Annotated[str, "Explain your reasoning for the score based on the answer and context"]

groundedness_prompt = """
You are a teacher grading whether a chatbot's answer is grounded in the provided context documents.
You will be given:
- The chatbot's answer.
- A set of retrieved/context documents.

Determine if the answer is based on and supported by the context provided. Mark it as 'grounded' if the information in the answer matches, quotes, or meaningfully relies on the content of the retrieved documents. If the answer introduces new information not present in the context, mark it as not grounded.

Return your evaluation in the following JSON format:
{
  "grounded": true or false,
  "explanation": "Briefly explain why the answer is grounded or not, referring to specific statements in the answer and context."
}
"""

groundedness_llm = init_chat_model(model="gpt-4o-mini", temperature=0).with_structured_output(GroundednessGrade, method='json_schema', strict=True)

def groundedness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict = None) -> bool:
    """An evaluator for groundedness of answer to retrieved documents"""
    # rag_bot returns LangChain Documents under outputs["documents"], not inputs
    docs = "\n\n".join(
        getattr(d, "page_content", str(d)) for d in (outputs.get("documents") or [])
    )
    data = f"""\
        CHATBOT ANSWER: {outputs['answer']}
        RETRIEVED DOCUMENTS:
        {docs}
    """
    grader = groundedness_llm.invoke([
        {"role": "system", "content": groundedness_prompt},
        {"role": "user", "content": data},
    ])
    return grader['grounded']

In [56]:
### Retereiveal relevance : retrieved docs vs input 


from typing import TypedDict, Annotated

class RetrievalRelevanceGrade(TypedDict):
    relevant: Annotated[bool, "true if the retrieved documents are relevant to the input question, False otherwise"]
    explanation: Annotated[str, "Explain your reasoning for the score based on the input and retrieved docs"]

retrieval_relevance_prompt = """
You are a teacher grading whether the documents retrieved for a question are relevant to that question.
You will be given:
- The input question.
- A set of retrieved/context documents.

Determine if the retrieved documents are relevant and useful to answer the question. Mark them as 'relevant' if their content helps answer, directly addresses, or otherwise pertains to the question. If they do not address the question or are unrelated, mark as not relevant.

Return your evaluation in the following JSON format:
{
  "relevant": true or false,
  "explanation": "Briefly explain why the retrieved documents are relevant or not, referring to specific parts of the question and the documents."
}
"""

retrieval_relevance_llm = init_chat_model(model="gpt-4o-mini", temperature=0).with_structured_output(RetrievalRelevanceGrade, method='json_schema', strict=True)

def retrieval_relevance_evaluator(inputs: dict, outputs: dict, reference_outputs: dict = None) -> bool:
    """An evaluator for the relevance of retrieved documents to the input question"""
    docs = "\n\n".join(
        getattr(d, "page_content", str(d)) for d in (outputs.get("documents") or [])
    )
    data = f"""\
        QUESTION: {inputs.get('question', '')}
        RETRIEVED DOCUMENTS:
        {docs}
    """
    grader = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_prompt},
        {"role": "user", "content": data},
    ])
    return grader['relevant']

In [58]:
### Hallucination & coherence (extra RAG metrics)

from typing import TypedDict, Annotated

# no_hallucination: True means the model did NOT add important facts that aren't in the retrieved docs.
# coherence: True means the answer reads as one clear, non-contradictory response to the question.


class HallucinationGrade(TypedDict):
    has_hallucination: Annotated[
        bool,
        "true if the answer asserts specific facts, numbers, names, or claims not supported by (or contradicted by) the documents",
    ]
    explanation: Annotated[str, "Point to any unsupported or contradicted statements, or say none found"]


hallucination_prompt = """
You check for hallucinations in a RAG-style answer.

You receive:
- The user's question
- Retrieved context documents (the ONLY sources the model should rely on for specific facts)
- The model's answer

Mark has_hallucination TRUE if the answer includes concrete factual claims, statistics, named entities, or definitive statements that are NOT supported by the documents, or that clearly CONTRADICT the documents.
Mark FALSE if the answer stays within what the documents support, paraphrases them fairly, gives only high-level generic advice without unsupported specifics, or honestly says the context does not contain enough information.

Do NOT mark hallucination for reasonable paraphrase, synthesis clearly grounded in the text, or opinion-free summarization of the given context.
"""


class CoherenceGrade(TypedDict):
    coherent: Annotated[
        bool,
        "true if the answer is logically consistent, not self-contradictory, and plausibly addresses the question",
    ]
    explanation: Annotated[str, "Brief note on structure or contradictions"]


coherence_prompt = """
You grade coherence and clarity of a chatbot answer (not factual correctness against a reference).

Given the QUESTION and the ANSWER, decide if the answer is coherent: internally consistent, not contradictory, and on-topic for what was asked.
Mark coherent FALSE if the answer contradicts itself, is mostly gibberish, ignores the question, or is so fragmented it cannot be followed.
"""


hallucination_llm = init_chat_model(model="gpt-4o-mini", temperature=0).with_structured_output(
    HallucinationGrade, method="json_schema", strict=True
)
coherence_llm = init_chat_model(model="gpt-4o-mini", temperature=0).with_structured_output(
    CoherenceGrade, method="json_schema", strict=True
)


def no_hallucination_evaluator(
    inputs: dict, outputs: dict, reference_outputs: dict = None
) -> bool:
    """True if the answer avoids unsupported factual claims vs retrieved docs (low hallucination)."""
    docs = "\n\n".join(
        getattr(d, "page_content", str(d)) for d in (outputs.get("documents") or [])
    )
    payload = f"""QUESTION: {inputs.get("question", "")}

RETRIEVED DOCUMENTS:
{docs}

MODEL ANSWER:
{outputs.get("answer", "")}
"""
    grader = hallucination_llm.invoke(
        [
            {"role": "system", "content": hallucination_prompt},
            {"role": "user", "content": payload},
        ]
    )
    return not grader["has_hallucination"]


def coherence_evaluator(
    inputs: dict, outputs: dict, reference_outputs: dict = None
) -> bool:
    """True if the answer is readable, consistent, and addresses the question."""
    payload = f"""QUESTION: {inputs.get("question", "")}

ANSWER:
{outputs.get("answer", "")}
"""
    grader = coherence_llm.invoke(
        [
            {"role": "system", "content": coherence_prompt},
            {"role": "user", "content": payload},
        ]
    )
    return grader["coherent"]


In [59]:
### Run the evalauation

def rag_target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])


experiment_results = client.evaluate(
    rag_target,
    data=rag_dataset_name,
    evaluators=[
        correctness_evaluator,
        relevance_evaluator,
        groundedness_evaluator,
        retrieval_relevance_evaluator,
        no_hallucination_evaluator,
        coherence_evaluator,
    ],
    experiment_prefix="rag-bot-evaluation",
    metadata={"version": "rag_bot + gpt-4o-mini evaluators"},
)

experiment_results.to_pandas()



View the evaluation results for experiment: 'rag-bot-evaluation-2915517f' at:
https://smith.langchain.com/o/625709e1-aba8-489e-9216-414289067b1e/datasets/5c9e953c-ccdb-44de-8b98-f970597c4452/compare?selectedSessions=8c0d6268-8d5e-4266-a1d5-e9ac21d0045d




4it [01:47, 26.96s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness_evaluator,feedback.relevance_evaluator,feedback.groundedness_evaluator,feedback.retrieval_relevance_evaluator,feedback.no_hallucination_evaluator,feedback.coherence_evaluator,execution_time,example_id,id
0,What are some types of agents based on their i...,Agents can be categorized based on how they in...,[page_content='categories of agents:There are ...,None,Types of agents based on interaction include i...,False,True,True,True,True,True,12.623278,26fc7ee5-2991-4533-943b-8fc86d5ffd00,019d3a84-acdb-70e2-881b-5dd1c7d52742
1,What is the difference between single agent an...,The primary difference between single agent sy...,"[page_content='analyze data for insights, opti...",None,A single agent system features one agent opera...,True,True,True,True,True,True,11.264286,72f11b5f-e817-438e-8fea-509250c9147d,019d3a85-2e08-7e11-8189-9379f8bf7086
2,What are AI agents and how do they differ from...,AI agents are sophisticated software systems d...,[page_content='AI agents have the highest degr...,None,AI agents are autonomous or semi-autonomous en...,True,True,True,True,True,True,9.546729,ac96e0d6-cf4e-422e-917d-0bc72c15cec9,019d3a85-9b72-7293-9bab-2af2e6f85cb0
3,Why are multi-agent systems important in AI?,Multi-agent systems are important in AI for se...,[page_content='and roles of individual agents ...,None,Multi-agent systems are important because they...,True,True,True,True,True,True,9.358577,f90bb2f8-35c3-4636-89f0-783ad5b281d5,019d3a85-f212-7e50-8d5f-f688ee9229b3
